# Checks notebook

This notebooks is used to:
- make checks between species and occurrence tables
- make counts for tables in the manuscript
- prepares Table 1
- calculates Simpson index for species in references
- prepares table for QGIS

### 1. Read files

We will read two tables
- taxon
- occurrences

In [62]:
import pandas as pd
import os

df_species = pd.read_csv('../dados_raw/checklist-species-29Abr2026.csv')

# Filter for taxonRank not genus
df_species.shape

(4440, 19)

### 2. Analyse **Taxon** table

In [63]:
# Count number of species (taxonID) per taxonomic status class (taxonomicStatus)
species_count_per_status = df_species.groupby('taxonomicStatus')['taxonID'].count()
print(species_count_per_status)
print("\nTotal number of species:", df_species.shape[0])

taxonomicStatus
accepted    2012
invalid       19
synonym     2409
Name: taxonID, dtype: int64

Total number of species: 4440


### 2.1. Count number of taxa per taxonRank
- Table: taxon
- description: includes all taxa status: accepted, synonum, invalid

In [64]:
# Count number of species (taxonID) per taxonRank in df_species
species_count_per_rank_total = df_species.groupby('taxonRank')['taxonID'].count()
print(species_count_per_rank_total)
print("\nTotal number of species:", df_species.shape[0])

taxonRank
forma          199
species       3605
subspecies      50
variety        586
Name: taxonID, dtype: int64

Total number of species: 4440


List count for ranks above species

In [65]:
import numpy as np

higher_ranks = ['phylum', 'class', 'order', 'family', 'genus']

# ------------------------------------------------------------------
# 1.  Replace the “blank” and “Incertae sedis” values with NaN
#     so that they are automatically ignored by nunique(dropna=True)
# ------------------------------------------------------------------
ignore_vals = ['', 'Incertae sedis']

# Keep the original data intact – just work on a copy
df_species_clean  = df_species[higher_ranks].replace(ignore_vals, np.nan)

# ------------------------------------------------------------------
# 2.  Count unique taxa (NaN values are discarded)
# ------------------------------------------------------------------
total_rank_counts = df_species_clean.nunique(dropna=True)
print("Total taxa counts by rank:")
print(total_rank_counts)

Total taxa counts by rank:
phylum      2
class      12
order      61
family    140
genus     588
dtype: int64


Combine total_rank_counts and species_count_per_rank in a single table with the rank order phylum', 'class', 'order', 'family', 'genus', species, subspecies, variety, forma

In [66]:
import pandas as pd

# Define the order of ranks
rank_order = ['phylum', 'class', 'order', 'family', 'genus', 'species', 'subspecies', 'variety', 'forma']

# Combine the counts into a dictionary
combined_counts = {}
for rank in rank_order:
    if rank in total_rank_counts.index:
        combined_counts[rank] = total_rank_counts[rank]
    elif rank in species_count_per_rank_total.index:
        combined_counts[rank] = species_count_per_rank_total[rank]
    else:
        combined_counts[rank] = 0  # If not present, set to 0

# Create a DataFrame
combined_table = pd.DataFrame(list(combined_counts.items()), columns=['Rank', 'Count'])

print(combined_table)

         Rank  Count
0      phylum      2
1       class     12
2       order     61
3      family    140
4       genus    588
5     species   3605
6  subspecies     50
7     variety    586
8       forma    199


### 2.2 Count number of taxa per taxonRank for accepted

In [67]:
df_species_accepted = df_species[df_species['taxonomicStatus'].str.lower() == 'accepted'].reset_index(drop=True)
df_species_accepted.shape

(2012, 19)

In [68]:
# Count number of species (taxonID) per taxonRank in df_species filtered for taxonomicStatus == 'accepted'
species_count_per_rank_accepted = df_species_accepted.groupby('taxonRank')['taxonID'].count()
print(species_count_per_rank_accepted)
print("\nTotal number of species:", df_species_accepted.shape[0])

taxonRank
forma           47
species       1843
subspecies      20
variety        102
Name: taxonID, dtype: int64

Total number of species: 2012


In [69]:
import numpy as np

higher_ranks = ['phylum', 'class', 'order', 'family', 'genus']

# ------------------------------------------------------------------
# 1.  Replace the “blank” and “Incertae sedis” values with NaN
#     so that they are automatically ignored by nunique(dropna=True)
# ------------------------------------------------------------------
ignore_vals = ['', 'Incertae sedis']

# Keep the original data intact – just work on a copy
accepted_clean  = df_species_accepted[higher_ranks].replace(ignore_vals, np.nan)

# ------------------------------------------------------------------
# 2.  Count unique taxa (NaN values are discarded)
# ------------------------------------------------------------------
accepted_rank_counts = accepted_clean.nunique(dropna=True)
print("Accepted taxa counts by rank:")
print(accepted_rank_counts)


Accepted taxa counts by rank:
phylum      2
class      11
order      56
family    127
genus     483
dtype: int64


In [70]:
import pandas as pd

# Define the order of ranks
rank_order = ['phylum', 'class', 'order', 'family', 'genus', 'species', 'subspecies', 'variety', 'forma']

# Combine the counts into a dictionary
combined_counts_accepted = {}
for rank in rank_order:
    if rank in accepted_rank_counts.index:
        combined_counts_accepted[rank] = accepted_rank_counts[rank]
    elif rank in species_count_per_rank_accepted.index:
        combined_counts_accepted[rank] = species_count_per_rank_accepted[rank]
    else:
        combined_counts_accepted[rank] = 0  # If not present, set to 0

# Create a DataFrame
combined_table = pd.DataFrame(list(combined_counts_accepted.items()), columns=['Rank', 'Count'])

print(combined_table)

         Rank  Count
0      phylum      2
1       class     11
2       order     56
3      family    127
4       genus    483
5     species   1843
6  subspecies     20
7     variety    102
8       forma     47


### Count number of infragenus taxa for each genus

In [71]:
"""
PROMPT: 
aggregate df_species_accepted by genus and count the number of unique values of acceptedNameUsageID. Sort results in descending order 
"""

genus_taxa_counts = (
    df_species_accepted
    .groupby('genus')['taxonID']
    .nunique(dropna=True)
    .sort_values(ascending=False)
    .reset_index(name='unique_taxonID')
)

genus_taxa_counts

,genus,unique_taxonID
0,Lecanora,87
1,Cladonia,78
2,Rinodina,66
3,Lecidea,44
4,Ramalina,40
...,...,...
478,Megalospora,1
479,Erioderma,1
480,Epithamnolia,1
481,Epiphloea,1


find number of genus with only one infragenus taxon

In [72]:
one_taxon_genus_count = (genus_taxa_counts['unique_taxonID'] == 1).sum()
one_taxon_genus_count

np.int64(219)

### 2.3 Count number of taxa per taxonRank for synonym

In [73]:
df_species_synonym = df_species[df_species['taxonomicStatus'].str.lower() == 'synonym'].reset_index(drop=True)
df_species_synonym.shape

(2409, 19)

In [74]:
# Count number of species (taxonID) per taxonRank in df_species filtered for taxonomicStatus == 'synonym'
species_count_per_rank_synonym = df_species_synonym.groupby('taxonRank')['taxonID'].count()
print(species_count_per_rank_synonym)
print("\nTotal number of species:", df_species_synonym.shape[0])

taxonRank
forma          151
species       1746
subspecies      30
variety        482
Name: taxonID, dtype: int64

Total number of species: 2409


In [75]:
higher_ranks = ['phylum', 'class', 'order', 'family', 'genus']

# ------------------------------------------------------------------
# 1.  Replace the “blank” and “Incertae sedis” values with NaN
#     so that they are automatically ignored by nunique(dropna=True)
# ------------------------------------------------------------------
ignore_vals = ['', 'Incertae sedis']

# Keep the original data intact – just work on a copy
synonym_clean   = df_species_synonym[higher_ranks].replace(ignore_vals, np.nan)

# ------------------------------------------------------------------
# 2.  Count unique taxa (NaN values are discarded)
# ------------------------------------------------------------------
synonym_rank_counts = synonym_clean.nunique(dropna=True)
print("Synonym taxa counts by rank:")
print(synonym_rank_counts)


Synonym taxa counts by rank:
phylum      2
class      12
order      43
family     92
genus     271
dtype: int64


In [76]:
import pandas as pd

# Define the order of ranks
rank_order = ['phylum', 'class', 'order', 'family', 'genus', 'species', 'subspecies', 'variety', 'forma']

# Combine the counts into a dictionary
combined_counts_synonym = {}
for rank in rank_order:
    if rank in synonym_rank_counts.index:
        combined_counts_synonym[rank] = synonym_rank_counts[rank]
    elif rank in species_count_per_rank_synonym.index:
        combined_counts_synonym[rank] = species_count_per_rank_synonym[rank]
    else:
        combined_counts_synonym[rank] = 0  # If not present, set to 0

# Create a DataFrame
combined_table = pd.DataFrame(list(combined_counts_synonym.items()), columns=['Rank', 'Count'])

print(combined_table)

         Rank  Count
0      phylum      2
1       class     12
2       order     43
3      family     92
4       genus    271
5     species   1746
6  subspecies     30
7     variety    482
8       forma    151


### 2.4 Count number of taxa per taxonRank for invalid

In [77]:
df_species_invalid = df_species[df_species['taxonomicStatus'].str.lower() == 'invalid'].reset_index(drop=True)
df_species_invalid.shape

(19, 19)

In [78]:
# Count number of species (taxonID) per taxonRank in df_species filtered for taxonomicStatus == 'invalid'
species_count_per_rank_invalid = df_species_invalid.groupby('taxonRank')['taxonID'].count()
print(species_count_per_rank_invalid)
print("\nTotal number of species:", df_species_invalid.shape[0])

taxonRank
forma       1
species    16
variety     2
Name: taxonID, dtype: int64

Total number of species: 19


## 3.1. Occurrences

Read occurrences table

This is the table of occurrences to be published through GBIF. The properties of the table are:
- does not contain taxa at genus level
- does contain taxa with taxonomic status "invalid"
- does contain taxa with province "Beira"

In [79]:
df_occ = pd.read_csv('../dados_raw/checklist_occurrences_29Abr2026.csv', low_memory=False)
print("Number of occurrences:", len(df_occ))

Number of occurrences: 43718


Check that no occurrences are classified as genus

In [80]:
# update df removing all rows where taxonRank is "genus"
print("Number of occcurrences classified at genus rank:", df_occ[df_occ['taxonRank'] == 'genus'].shape[0])

Number of occcurrences classified at genus rank: 0


Remove taxa invalid

In [81]:
# exclude invalids to count only accepted and synonym

df_occ_valid_taxa = df_occ[(df_occ['taxonomicStatus'].str.lower() == 'accepted') | (df_occ['taxonomicStatus'].str.lower() == 'synonym')].reset_index(drop=True)
print("Number of valid taxa in df_occ_valid_taxa:", len(df_occ_valid_taxa))

Number of valid taxa in df_occ_valid_taxa: 43683


Confirm the number of accepted taxa represented in the valid occurrences table. Should be equal to the accepted taxa in taxa table

In [82]:
# Check if n_unique_taxa_occ equals the number of accepted taxa in df_species
n_unique_taxa_occ = df_occ_valid_taxa['acceptedNameUsageID'].nunique()
print("n_unique_taxa_occ:", n_unique_taxa_occ)
print("Number of accepted taxa in df_species:", len(df_species_accepted))
print("Are they equal?", n_unique_taxa_occ == len(df_species_accepted))

n_unique_taxa_occ: 2012
Number of accepted taxa in df_species: 2012
Are they equal? True


In [83]:
# create a list of values of taxonID in df_species_accepted that are not present in acceptedNameUsageID of df_occ_valid_taxa

missing_taxonids = (
    df_species_accepted.loc[
        ~df_species_accepted['taxonID'].isin(df_occ_valid_taxa['acceptedNameUsageID']),
        'taxonID'
    ]
    .unique()
    .tolist()
)

missing_taxonids

[]

In [84]:
# create a list of values of acceptedNameUsageID of df_occ_valid_taxa that are not present in taxonID in df_species_accepted

missing_taxonids = (
    df_occ_valid_taxa.loc[
        ~df_occ_valid_taxa['acceptedNameUsageID'].isin(df_species_accepted['taxonID']),
        'acceptedNameUsageID'
    ]
    .unique()
    .tolist()
)

missing_taxonids

[]

In [85]:
# create a list of values of taxonID in df_species_accepted that are not present in acceptedNameUsageID of df_occ_valid_taxa

### 3.1. Top species with highest number of occurrences

In [86]:
"""
PROMPT:
based on the df_occ_vali_taxa, list the top 10 species with the highest number of occurrences
"""

top10_species = (
    df_occ_valid_taxa
    .groupby('acceptedNameUsage')['occurrenceID']
    .count()
    .sort_values(ascending=False)
    .head(20)
    .reset_index(name='Occurrences')
)

top10_species

,acceptedNameUsage,Occurrences
0,Collema furfuraceum (Arnold) Du Rietz,595
1,Flavoparmelia caperata (L.) Hale,432
2,Ramalina farinacea (L.) Ach.,407
3,Lobaria pulmonaria (L.) Hoffm.,394
4,Evernia prunastri (L.) Ach.,378
5,Nephroma laevigatum Ach.,335
6,Ramalina fastigiata (Pers.) Ach.,332
7,Collema subflaccidum Degel.,297
8,Lobarina scrobiculata (Scop.) Nyl. ex Cromb.,296
9,"Pectenia plumbea (Lightf.) P.M. Jørg., L. Lind...",291


In [90]:

# Count occurrences per taxon
taxon_counts = df_occ_valid_taxa['acceptedNameUsageID'].value_counts()

# Get the acceptedNameUsageID for taxa with exactly 1 occurrence (singletons)
singleton_ids = taxon_counts[taxon_counts == 1].index

# Filter df_occ_valid_taxa for these IDs and get unique acceptedNameUsage and taxonRank
singleton_species = df_occ_valid_taxa[df_occ_valid_taxa['acceptedNameUsageID'].isin(singleton_ids)][['acceptedNameUsage', 'taxonRank']].drop_duplicates()

# Create a DataFrame for export
# singleton_df = pd.DataFrame({'Species': singleton_species})

# Export to CSV
singleton_species.to_csv('../dados_process/singletons.csv', index=False)

### 3.2. Table with counts per region

To build this table we remove what cannot be classified in a province. Corresponds to Table 1 of the manuscript.

In [ ]:
# update df removing all rows where taxonRank is "genus"
df_prov_tidy = df_occ_valid_taxa[df_occ_valid_taxa['provincia_process'] != 'Beira']
df_prov_tidy.shape

In [ ]:
"""
PROMPT:
create a table based on df_occ_valid_taxa, aggregated by column 'province_process' with the following columns

- name of province, named 'Province'
- number of occorrences in that province, named 'Occurrences'
- number of taxa in that province, i.e, unique values in column 'acceptedNameUsageID' in that province, named 'Taxa'
- percentage of taxa, i.e., the rate of the number of taxa in that province by the total number of taxa, named '% taxa', 
- number of singletons, i.e, the number of taxa that occurs only once in that province, named 'Singletons'
- number of doubleletons, i.e, the number of taxa that occurs only twice in that province, named 'Doubletons'
- number of publications in that province, i.e., number of unique values of 'id_reference_biblio', named 'Number of publications'
- the maximum number of taxa in a publication for that province, named 'Highest number of taxa in a single publication'

"""


import pandas as pd

# Total number of taxa
total_taxa = df_prov_tidy['acceptedNameUsageID'].nunique()

# Group by province_process
grouped = df_prov_tidy.groupby('provincia_process')

# Initialize a list to hold the results
results = []

for province, group in grouped:
    # Province name
    province_name = province
    
    # Number of occurrences
    occurrences = len(group)
    
    # Number of taxa (unique acceptedNameUsageID)
    taxa = group['acceptedNameUsageID'].nunique()
    
    # Percentage of taxa
    pct_taxa = (taxa / total_taxa) * 100 if total_taxa > 0 else 0
    
    # Count occurrences per taxon in this province
    taxon_counts = group['acceptedNameUsageID'].value_counts()
    
    # Singletons: taxa with exactly 1 occurrence
    singletons = (taxon_counts == 1).sum()
    
    # Doubletons: taxa with exactly 2 occurrences
    doubletons = (taxon_counts == 2).sum()
    
    # Number of publications (unique id_referencia_biblio)
    publications = group['id_referencia_biblio'].nunique()
    
    # Highest number of taxa in a single publication
    if publications > 0:
        # Group by publication within this province, count unique taxa per publication
        pub_taxa_counts = group.groupby('id_referencia_biblio')['acceptedNameUsageID'].nunique()
        highest_taxa_pub = pub_taxa_counts.max()
    else:
        highest_taxa_pub = 0
    
    # Append to results
    results.append({
        'Province': province_name,
        'Occurrences': occurrences,
        'Taxa': taxa,
        '% taxa': pct_taxa,
        'Singletons': singletons,
        'Doubletons': doubletons,
        'Number of publications': publications,
        'Highest number of taxa in a single publication': highest_taxa_pub
    })

# Create DataFrame from results
province_table = pd.DataFrame(results)
province_table.sort_values(by='Occurrences', ascending=False, inplace=True)
province_table.reset_index(drop=True, inplace=True)
province_table

In [ ]:
"""
PROMPT:
create a table with the same columns as province_table, based on df_occ_valid_taxa, but for the total table, without aggregation by province
"""


import pandas as pd

# Total number of taxa
total_taxa = df_occ_valid_taxa['acceptedNameUsageID'].nunique()

# Province name for total
province_name = "Total"

# Number of occurrences
occurrences = len(df_occ_valid_taxa)

# Number of taxa
taxa = total_taxa

# Percentage of taxa
pct_taxa = 100.0

# Count occurrences per taxon
taxon_counts = df_occ_valid_taxa['acceptedNameUsageID'].value_counts()

# Singletons: taxa with exactly 1 occurrence
singletons = (taxon_counts == 1).sum()

# Doubletons: taxa with exactly 2 occurrences
doubletons = (taxon_counts == 2).sum()

# Number of publications (unique id_referencia_biblio)
publications = df_occ_valid_taxa['id_referencia_biblio'].nunique()

# Highest number of taxa in a single publication
pub_taxa_counts = df_occ_valid_taxa.groupby('id_referencia_biblio')['acceptedNameUsageID'].nunique()
highest_taxa_pub = pub_taxa_counts.max()

# Create DataFrame
total_table = pd.DataFrame([{
    'Province': province_name,
    'Occurrences': occurrences,
    'Taxa': taxa,
    '% taxa': pct_taxa,
    'Singletons': singletons,
    'Doubletons': doubletons,
    'Number of publications': publications,
    'Highest number of taxa in a single publication': highest_taxa_pub
}])

total_table

### 3.3 Count number of taxa that occur in all provinces 

In [ ]:
# in df_prov_tidy, count the number of species (unique acceptedNameUsageID) that occur in all 11 provinces
count_prov_per_taxa = df_prov_tidy.groupby('acceptedNameUsageID')['provincia_process'].nunique()
count_taxa_all_provinces = (count_prov_per_taxa == 11).sum()
print(f"Number of taxa that occur in all 11 provinces: {count_taxa_all_provinces}")




## 3.4 Number of taxa that are referred in only one publication

In [ ]:
# count number of unique references per species
ref_counts = df_prov_tidy.groupby("acceptedNameUsageID")["id_referencia_biblio"].nunique()

# select species that appear in only one reference
species_single_ref = ref_counts[ref_counts == 1]

# count them
n_species_single_ref = species_single_ref.shape[0]

print(n_species_single_ref)

In [ ]:
ref_counts

In [ ]:
taxon_counts

In [ ]:
import seaborn as sns
from scipy.stats import pearsonr

import matplotlib.pyplot as plt

# numeric columns from province_table
num_cols = [
  'Occurrences',
  'Taxa',
  '% taxa',
  'Singletons',
  'Doubletons',
  'Number of publications',
  'Highest number of taxa in a single publication'
]

df = province_table[num_cols].copy()

# PairGrid for biplot-like scatterplots and correlations
g = sns.PairGrid(df, diag_sharey=False)
g.map_lower(sns.regplot, scatter_kws={'s': 40, 'alpha': 0.7}, line_kws={'color': 'red'})
g.map_diag(sns.histplot, kde=False, color='lightgray')

def corr_annot(x, y, **kwargs):
  r, p = pearsonr(x, y)
  ax = plt.gca()
  ax.annotate(f"r={r:.2f}\np={p:.3f}", xy=(0.05, 0.90), xycoords=ax.transAxes, fontsize=9)

g.map_upper(corr_annot)

g.fig.suptitle("Province table pairwise biplots with Pearson correlation", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
df_prov_tidy

## Verificar o efeito de dominância de uma referência

✅ Best index for your question: Pielou’s evenness (recommended)

If your goal is:

“Are the occurrences in this province supported evenly by many references, or dominated by one/few papers?”

then Pielou’s evenness 
J
J is probably the most interpretable.

It is based on Shannon entropy and ranges from:

1 → perfectly even
all references contribute similarly
0 → highly uneven
one reference dominates the records

Formula:

J=H′ln⁡(S)
J=
ln(S)
H
′
	​


Where:

H′=−∑piln⁡pi
H
′
=−∑p
i
	​

lnp
i
	​

pi
p
i
	​

 = proportion of occurrences from reference 
i
i
S
S = number of references in that province

✅ Alternative: Simpson concentration (excellent for dominance)

If your interest is specifically:

“Is a province mostly based on a single source?”

then Simpson concentration / Herfindahl index may be even better:

D=∑pi2
D=∑p
i
2
	​


Interpretation:

high 
D
D → dominated by one/few references
low 
D
D → many references contribute similarly

This index gives extra weight to dominant references, which is often ideal for bibliography bias.

In [ ]:
taxa_by_ref_province = (
  df_prov_tidy
  .groupby(['provincia_process', 'id_referencia_biblio'])['acceptedNameUsageID']
  .nunique()
  .reset_index(name='taxa_count')
  .sort_values(['provincia_process', 'taxa_count'], ascending=[True, False])
)

taxa_by_ref_province.to_csv(
  '../dados_process/num_taxa_per_reference_province.csv',
  index=False
)

taxa_by_ref_province.head(20)

In [ ]:
import numpy as np

def pielou_evenness(group):
    p = group["taxa_count"] / group["taxa_count"].sum()
    H = -(p * np.log(p)).sum()
    S = len(group)
    return H / np.log(S) if S > 1 else 0

pielou_by_province = (
    taxa_by_ref_province
    .groupby("provincia_process")
    .apply(pielou_evenness)
    .reset_index(name="pielou_J")
)

print(pielou_by_province)

In [ ]:
def simpson_concentration(group):
    p = group["taxa_count"] / group["taxa_count"].sum()
    return (p ** 2).sum()

simpson_by_province = (
    taxa_by_ref_province
    .groupby("provincia_process")
    .apply(simpson_concentration)
    .reset_index(name="simpson_D")
)

print(simpson_by_province)

In [ ]:
province_ref_evenness = simpson_by_province.merge(
    pielou_by_province,
    on="provincia_process"
)

print(province_ref_evenness)

In [ ]:
combined_province_analysis = province_table.merge(
  province_ref_evenness,
  left_on='Province',
  right_on='provincia_process'
).drop('provincia_process', axis=1)

combined_province_analysis

Calculate Simpson for the whole country

In [ ]:
country_by_ref = (
  df_prov_tidy
  .groupby('id_referencia_biblio')['acceptedNameUsageID']
  .nunique()
  .reset_index(name='taxa_count')
  .sort_values(['taxa_count'], ascending=[False])
)

country_by_ref.head(20)

In [ ]:
p = country_by_ref["taxa_count"] / country_by_ref["taxa_count"].sum()
simpson_country = (p ** 2).sum()

print(simpson_country)

In [ ]:
import numpy as np

H = -(p * np.log(p)).sum()
S = len(country_by_ref)

pielou_country = H / np.log(S) if S > 1 else 0

print(pielou_country)

## Prepare tables for QGIS 

Prepare table with counts of taxa and occorrences per province

In [ ]:
# Use table that does not include Beira province
df_prov_tidy.describe()

In [ ]:
df_prov_tidy_clean = df_prov_tidy.dropna(subset=['provincia_process']).copy()

In [ ]:
# Create a table aggregated by province with occurrence and taxa counts
province_summary = (
  df_prov_tidy_clean
  .groupby('provincia_process')
  .agg(
    occurrences=('occurrenceID', 'nunique'),
    taxa=('acceptedNameUsageID', 'nunique')
  )
  .reset_index()
  .rename(columns={'provincia_process': 'Province'})
  .sort_values('occurrences', ascending=False)
)

province_summary

In [ ]:
# read table with ID of province
province_ids = pd.read_csv('../dados_process/province_ids.csv')

In [ ]:
# merge with province_summary to get IDs
province_summary_with_ids = province_summary.merge(
  province_ids,
  left_on='Province',
  right_on='provincia_process',
  how='left'
).drop('provincia_process', axis=1)

province_summary_with_ids

In [ ]:
import geopandas as gpd

# Open the geopackage
gdf = gpd.read_file('../qgis/dados_process/provincias_flora.gpkg')

# Display basic information
print(gdf.head())
print(gdf.info())

In [ ]:
# merge with gdf
gdf_provinces = gdf.merge(
  province_summary_with_ids,
  on='PROVID'
).drop('Province', axis=1)

gdf_provinces

In [ ]:
# write to geopackage
gdf_provinces.to_file('../qgis/dados_process/provincias_flora_summary.gpkg', driver='GPKG') 